# Notebook 04 — Climate Exposure Linkage: Connecting Health Records to the Environment
**GeoAI for Climate, Environment and Health**

---

> **Where you are in the pipeline**
>
> ```
> 📋 Health records → 🛰 Satellite → 🌍 Environment → [🔗 Link] → ⏱ Timing → 🎯 Results & GeoAI
> ```
>
> You have birth records. You have explored the climate data and understand its provenance.
> They still live in separate tables with no common ID — only coordinates.
> This notebook performs **climate exposure linkage**: for every birth record,
> find the nearest climate grid point and attach its values.
>
> In GIS this is called a **spatial join**. In epidemiology it is called
> **exposure assignment** or **exposure linkage** — the step that makes
> it possible to ask *"what climate did this mother experience during pregnancy?"*

---


In [ ]:
%%html
<div id="nb04-prereq-gate" style="font-family:system-ui,sans-serif;margin:0 0 16px"></div>
<script>
(function(){
  var PREV_KEY='nb03_passed';
  var PREV_LABEL='Module 3: Environmental Datasets';
  var gate=document.getElementById('nb04-prereq-gate');
  if(!gate)return;
  if(!localStorage.getItem(PREV_KEY)){
    gate.innerHTML='<div style="background:#fff3cd;border:1px solid #ffc107;border-left:5px solid #ffc107;border-radius:6px;padding:14px 18px;color:#664d03"><b>&#9888; Prerequisite not detected</b><br>It looks like you have not yet passed the quiz for <b>'+PREV_LABEL+'</b>. We recommend completing that module first before continuing here. Your facilitator can help if needed.</div>';
  }else{
    gate.innerHTML='<div style="background:#d1e7dd;border-left:5px solid #198754;border-radius:6px;padding:10px 16px;color:#0a3622"><b>&#10003; '+PREV_LABEL+' completed</b> &#8212; welcome to the next module.</div>';
  }
})();
</script>

---
## Section 1 — Load Data

We load four tables: the monthly climate records and cluster metadata (both from Notebook 02), the raw birth records, and the health facility register. All four are needed by different sections of this notebook. Run this cell before any other — variables `climate`, `clusters`, `births`, and `facilities` must be in scope.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.spatial import cKDTree
from scipy import stats

import pathlib, sys

def _find_data():
    here = pathlib.Path().resolve()
    for p in (here, here.parent, here.parent.parent):
        if (p / 'data').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))  # makes shared_utils importable
            return p / 'data'
    raise FileNotFoundError('Cannot find data/ directory')

DATA_DIR = _find_data()
print(f'Data directory: {DATA_DIR}')

def mc_question(question, options, correct, explanation):
    try:
        import ipywidgets as w
        from IPython.display import display, HTML
        header = w.HTML(f'<div style="font-size:15px;font-weight:600;margin-bottom:10px">🧠 {question}</div>')
        radio = w.RadioButtons(options=[f"{k})  {v}" for k, v in options.items()],
                               layout=w.Layout(width="100%"),
                               style={"description_width": "0px"})
        feedback = w.Output()
        def _check(change):
            if change["name"] != "value": return
            letter = change["new"][0]
            with feedback:
                feedback.clear_output(wait=True)
                if letter == correct:
                    display(HTML(f'<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #28a745">✅ <b>Correct!</b>&nbsp; {explanation}</div>'))
                else:
                    display(HTML(f'<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b>&nbsp; The answer is <b>{correct}</b>. {explanation}</div>'))
        radio.observe(_check, names="value")
        display(w.VBox([header, radio, feedback],
                       layout=w.Layout(border="1px solid #dee2e6", padding="16px 20px",
                                       border_radius="8px", margin="16px 0")))
    except ImportError:
        print(f"Q: {question}")
        for k, v in options.items():
            print(f"  {k}) {v}{' ← CORRECT' if k == correct else ''}")
        print(f"Explanation: {explanation}")


In [ ]:
# Load all data tables needed for this notebook
climate    = pd.read_csv(DATA_DIR / 'zw_climate_monthly.csv')
clusters   = pd.read_csv(DATA_DIR / 'zw_clusters.csv')
births     = pd.read_csv(DATA_DIR / 'zw_births_raw.csv')
facilities = pd.read_csv(DATA_DIR / 'health_facilities.csv')
facilities = facilities.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

print(f"Climate records : {len(climate):,}  ({climate['cluster_id'].nunique()} clusters × {climate['year_month'].nunique() if 'year_month' in climate.columns else climate.groupby(['year','month']).ngroups} months)")
print(f"Clusters        : {len(clusters):,}")
print(f"Births          : {len(births):,}")
print(f"Facilities      : {len(facilities):,}")
print()
print("Births columns:", births.columns.tolist())

**Why annual means?** The monthly climate table has 60 rows per cluster (2011–2015). For the spatial geometry exercises in this notebook — building the KD-tree, measuring jitter uncertainty, and checking spatial confounding — we only need one representative point per cluster. We therefore collapse the monthly records to a single annual mean per cluster. This does **not** mean we will analyse exposure at annual resolution: Notebook 04 returns to the monthly data to construct trimester-specific windows. Here we are just building the spatial index.

In [ ]:
# Annual mean climate per cluster — average all months, 2011–2015
clim_annual = (
    climate
    .groupby('cluster_id')[['temp_c', 'rain_mm', 'ndvi', 'dust_aod550', 'pm25']]
    .mean()
    .reset_index()
    .rename(columns={
        'temp_c':       'mean_temp',
        'rain_mm':      'mean_rain',
        'ndvi':         'mean_ndvi',
        'dust_aod550':  'mean_dust',
        'pm25':         'mean_pm25',
    })
)

# Attach cluster coordinates for spatial operations
# (cluster_lat/lon already in climate CSV but merge ensures alignment)
clim_annual = clim_annual.merge(
    clusters[['cluster_id', 'cluster_lat', 'cluster_lon', 'province', 'urban']],
    on='cluster_id'
)

print(f"Exposure grid: {len(clim_annual)} clusters")
print(clim_annual[['mean_temp', 'mean_rain', 'mean_ndvi', 'mean_pm25']].describe().round(2))

**Why cKDTree?** A naive nearest-neighbour search compares every query point against every grid point — O(n×m) operations. With 399 clusters and thousands of births or facilities, that is slow and scales badly. `cKDTree` partitions coordinates into a tree structure at build time; each subsequent query then runs in O(log n), making the lookup near-instantaneous regardless of how many points you query.

**Distance in geographic coordinates.** The KD-tree works in raw latitude/longitude space, which is fine for finding the *index* of the nearest point. To convert the result into kilometres we use the Haversine formula (or the flat-earth approximation below for short distances). The function defined in the next cell will be reused throughout this notebook.

---
## Section 2 — Worked Example: Nearest-Neighbour Exposure Linkage

In **environmental epidemiology**, the step of assigning a satellite-derived climate value
to each participant based on where they lived is called **exposure linkage** or
**climate exposure assignment**. The underlying GIS operation is a **spatial join**.

We use a `cKDTree` nearest-neighbour algorithm — it builds a spatial index in milliseconds
and can query 5,794 birth locations against 399 cluster centroids in a single call.

We first demonstrate the method by finding the nearest DHS cluster to each Zimbabwe health
facility, then apply the same logic to link births to their cluster's annual climate values.


In [ ]:
def haversine_approx_km(lat1, lon1, lat2, lon2):
    """Flat-earth approximation. Accurate to <0.5% for distances <200 km."""
    mid_lat  = np.radians((np.asarray(lat1) + np.asarray(lat2)) / 2.0)
    dlat_km  = (np.asarray(lat2) - np.asarray(lat1)) * 111.0
    dlon_km  = (np.asarray(lon2) - np.asarray(lon1)) * 111.0 * np.cos(mid_lat)
    return np.sqrt(dlat_km**2 + dlon_km**2)

In [ ]:
# Build k-d tree on cluster coordinates (lat, lon)
cluster_coords = clim_annual[['cluster_lat', 'cluster_lon']].values
tree = cKDTree(cluster_coords)

# Query: for each facility, find the nearest DHS cluster
facility_coords = facilities[['latitude', 'longitude']].values
_, nn_idx = tree.query(facility_coords, k=1)

nn_rows = clim_annual.iloc[nn_idx].reset_index(drop=True)

result = pd.DataFrame({
    'facility_id'     : facilities['facility_id'].values,
    'facility_type'   : facilities['facility_type'].values,
    'fac_district'    : facilities['district'].values,
    'nearest_cluster' : nn_rows['cluster_id'].values,
    'cluster_province': nn_rows['province'].values,
    'distance_km'     : haversine_approx_km(
        facilities['latitude'].values,  facilities['longitude'].values,
        nn_rows['cluster_lat'].values,  nn_rows['cluster_lon'].values
    ).round(2)
})

print("Facility → nearest DHS cluster (first 10 rows):")
print(result.head(10).to_string(index=False))
print()
print(f"Median facility-to-cluster distance: {result['distance_km'].median():.1f} km")
print(f"Max    facility-to-cluster distance: {result['distance_km'].max():.1f} km")

In [ ]:
import contextily as ctx
from pyproj import Transformer as _T

_proj = _T.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# Project cluster and facility coordinates to Web Mercator
cx, cy = _proj.transform(clim_annual['cluster_lon'].values, clim_annual['cluster_lat'].values)
clim_annual = clim_annual.copy()
clim_annual['_mx'] = cx
clim_annual['_my'] = cy

fx, fy = _proj.transform(facilities['longitude'].values, facilities['latitude'].values)

fig, ax = plt.subplots(figsize=(9, 8))

sc = ax.scatter(
    clim_annual['_mx'], clim_annual['_my'],
    c=clim_annual['mean_temp'], cmap='RdYlBu_r',
    s=28, alpha=0.85, zorder=3, label=f'DHS cluster (n={len(clim_annual)})'
)
plt.colorbar(sc, ax=ax, label='Mean annual temperature (°C)', shrink=0.65)

ax.scatter(fx, fy, marker='*', s=200, color='black', zorder=5,
           edgecolors='white', linewidths=0.5,
           label=f'Health facility (n={len(facilities)})')

ax.set_title(
    'Zimbabwe DHS 2015 — Cluster Locations and Health Facilities\n'
    'Coloured by mean annual temperature (ERA5)',
    fontsize=12
)
ax.set_axis_off()
ax.legend(loc='lower right', fontsize=10)

try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=7)
except Exception:
    pass

plt.tight_layout()
plt.show()

print(f"DHS clusters: {len(clim_annual)}   |   Health facilities: {len(facilities)}")
print("The spatial join assigns each facility's climate to the nearest cluster centroid.")

---
## Section 3 — Exercise 1 (3 pts): Leave-One-Out Nearest-Neighbour Distances

For each DHS cluster, find the nearest *other* cluster (excluding itself). Compute the distribution of inter-cluster distances.

**Store:**
- `median_nn_dist_km` — float, 1 decimal place
- `max_nn_dist_km`    — float, 1 decimal place

Plot a histogram of inter-cluster distances. This analysis reveals the effective spatial resolution of the DHS sampling frame and informs how much the 10 km jitter can "bleed" an observation across cluster boundaries.

In [ ]:

# k=2: first neighbour is the point itself (distance = 0); second is the nearest *other* cluster
_, nn_idxs = tree.query(cluster_coords, k=2)

lat_arr = clim_annual['cluster_lat'].values
lon_arr = clim_annual['cluster_lon'].values

nn_km = haversine_approx_km(
    lat_arr,               lon_arr,
    lat_arr[nn_idxs[:, 1]], lon_arr[nn_idxs[:, 1]]
)

median_nn_dist_km = round(float(np.median(nn_km)), 1)
max_nn_dist_km    = round(float(np.max(nn_km)), 1)

print(f"Median nearest-neighbour distance: {median_nn_dist_km} km")
print(f"Max    nearest-neighbour distance: {max_nn_dist_km} km")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(nn_km, bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(median_nn_dist_km, color='firebrick', lw=1.8, linestyle='--',
           label=f'Median = {median_nn_dist_km} km')
ax.axvline(10, color='darkorange', lw=1.4, linestyle=':',
           label='DHS max jitter = 10 km')
ax.set_xlabel('Distance to nearest other cluster (km)', fontsize=11)
ax.set_ylabel('Number of clusters', fontsize=11)
ax.set_title('Inter-cluster nearest-neighbour distances — Zimbabwe DHS 2015', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()


**Interpretation (discuss with your group).** If the median nearest-neighbour distance substantially exceeds the DHS jitter radius (up to 10 km), then displaced centroids are unlikely to fall inside a neighbouring cluster's territory. If clusters are densely packed, jitter causes spatial misclassification: the displaced centroid absorbs the environmental signal of a different cluster, adding non-differential measurement error to any exposure–outcome association.

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 When we assign climate data from the nearest DHS cluster to each birth record, what key assumption are we making?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q1" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> All households in Zimbabwe are within 5 km of a DHS cluster</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q1" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> The climate at the DHS cluster location represents the climate experienced by the mother</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q1" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> DHS clusters are perfectly positioned at household centroids</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q1" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> Climate data has no spatial variation within 10 km</span></label></div>
  <div id="nb04q1-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="This is the core assumption of ecological exposure assignment: the nearest measured location is a reasonable proxy for unmeasured individual exposure. It introduces exposure misclassification — an important limitation to acknowledge.",ID="nb04q1";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Section 4 — Worked Example: The Jitter Uncertainty Envelope

DHS coordinates are displaced to protect respondent privacy. We simulate the displacement for rural clusters and measure how much the assigned exposure can vary purely as a function of positional uncertainty.

In [ ]:
np.random.seed(42)

N_SIM        = 100    # displaced positions simulated per cluster
RURAL_JITTER = 5.0   # km — DHS maximum displacement for rural clusters
DEG_PER_KM   = 1.0 / 111.0

rural_mask     = clim_annual['urban'] == 0
rural_clusters = clim_annual[rural_mask].reset_index(drop=True)
chosen         = rural_clusters.sample(10, random_state=42).reset_index(drop=True)

jitter_results = []
for _, row in chosen.iterrows():
    angles   = np.random.uniform(0, 2 * np.pi, N_SIM)
    radii_km = np.random.uniform(0, RURAL_JITTER, N_SIM)
    jit_lat  = row['cluster_lat'] + radii_km * np.cos(angles) * DEG_PER_KM
    jit_lon  = row['cluster_lon'] + radii_km * np.sin(angles) * DEG_PER_KM

    _, idx = tree.query(np.column_stack([jit_lat, jit_lon]), k=1)

    temp_vals = clim_annual['mean_temp'].iloc[idx].values
    rain_vals = clim_annual['mean_rain'].iloc[idx].values

    jitter_results.append({
        'cluster_id'  : int(row['cluster_id']),
        'true_temp'   : round(row['mean_temp'], 2),
        'temp_range'  : round(temp_vals.max() - temp_vals.min(), 2),
        'rain_range'  : round(rain_vals.max() - rain_vals.min(), 1),
    })

jitter_df = pd.DataFrame(jitter_results)
print("Jitter uncertainty — 100 simulated displacements per rural cluster (5 km radius):")
print(jitter_df.rename(columns={
    'temp_range': 'Temp range (°C)', 'rain_range': 'Rain range (mm/mo)'
}).to_string(index=False))
print()
print(f"Median temp uncertainty (±):  {jitter_df['temp_range'].median() / 2:.2f} °C")
print(f"Median rain uncertainty (±):  {jitter_df['rain_range'].median() / 2:.1f} mm/mo")
print()
print("Implication: positional jitter introduces measurable temperature uncertainty,")
print("particularly near elevation gradients and urban-rural boundaries.")

The table shows simulated displacement uncertainty using our gridded climate data. Because ERA5 has ~25 km grid spacing and CHIRPS ~5 km, a 5 km displacement often stays within the same grid cell — so the range appears near zero for many clusters in this pre-extracted dataset. In practice with finer-resolution products (e.g., MODIS LST at 1 km, daily CHIRPS at 5 km), a 5 km displacement near an elevation gradient or land-cover boundary would produce meaningful temperature or rainfall differences. This is **non-differential exposure misclassification**: it attenuates true associations toward the null and is an acknowledged limitation in all DHS-based environmental epidemiology.

---
## Section 5 — Exercise 2 (3 pts): Join Births to Cluster-Level Annual Mean Climate

Merge `births` with `clim_annual` on `cluster_id` using a left join. Confirm there are no unmatched rows. Store the joined DataFrame as `births_env`.

Then compute: what fraction of births belong to clusters where the mean annual temperature > 18 °C? Store as `frac_hot_clusters` (float, 3 decimal places).

In [ ]:
births_env = births.merge(
    clim_annual[['cluster_id', 'mean_temp', 'mean_rain', 'mean_ndvi', 'mean_pm25', 'mean_dust']],
    on='cluster_id',
    how='left'
)

unmatched = births_env['mean_temp'].isnull().sum()
print(f"Unmatched births (no climate record): {unmatched}")
assert unmatched == 0, "All births must match a cluster — check cluster_id values."

frac_hot_clusters = round(float((births_env['mean_temp'] > 18).mean()), 3)

print(f"Births joined successfully: {len(births_env):,} rows")
print(f"Fraction of births in clusters with mean annual temp > 18 °C: {frac_hot_clusters:.3f}")

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 A birth record is assigned climate data from a cluster 31 km away — the nearest available. What type of error does this introduce?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q2" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Systematic selection bias — certain groups were excluded from the DHS</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q2" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Exposure misclassification — the assigned climate values may not reflect the environment where the mother actually lived</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q2" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Confounding — another variable explains the association</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q2" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> Measurement error in birth weight recording</span></label></div>
  <div id="nb04q2-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="Distance-based misclassification dilutes real associations (attenuation bias). In practice, researchers often set a maximum acceptable distance and flag or exclude distant matches.",ID="nb04q2";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Section 6 — Worked Example: Spatial Confounding — Facility Proximity and LBW

Clusters near health facilities may have higher care-seeking rates and more complete birth-weight recording. This creates spatial confounding that could distort any climate–LBW association. We stratify clusters by distance to the nearest facility as a diagnostic check.

In [ ]:
PROXIMITY_KM = 10.0

fac_tree = cKDTree(facilities[['latitude', 'longitude']].values)
_, fac_idx = fac_tree.query(cluster_coords, k=1)

dist_to_fac = haversine_approx_km(
    lat_arr,
    lon_arr,
    facilities['latitude'].values[fac_idx],
    facilities['longitude'].values[fac_idx]
)

clim_annual = clim_annual.copy()
clim_annual['dist_to_fac_km'] = dist_to_fac
clim_annual['near_facility']  = dist_to_fac <= PROXIMITY_KM

cluster_compare = (
    clusters[['cluster_id', 'lbw_rate']]
    .merge(clim_annual[['cluster_id', 'near_facility']], on='cluster_id')
)

near   = cluster_compare[cluster_compare['near_facility']]
remote = cluster_compare[~cluster_compare['near_facility']]

print(f"Clusters within {PROXIMITY_KM} km of a facility   (n={len(near):3d}):  "
      f"mean LBW rate = {near['lbw_rate'].mean():.3f}")
print(f"Clusters beyond {PROXIMITY_KM} km of a facility   (n={len(remote):3d}):  "
      f"mean LBW rate = {remote['lbw_rate'].mean():.3f}")
print()
print("Note: this stratification conflates access, care quality, and recording completeness.")
print("It is a spatial diagnostic, not a causal estimate.")

If cluster-level LBW rates differ between facility-proximate and remote clusters, it could reflect genuine differences in care quality, or simply that facility-adjacent clusters have more complete birth-weight recording (low-birth-weight births in remote areas may be delivered at home and never weighed). Spatial confounding of this kind cannot be removed by adjusting for the environment alone — it requires explicitly controlling for facility access or restricting analysis to facility deliveries. Keep this in mind when interpreting any unadjusted climate-LBW correlation.

---
## Section 7 — Exercise 3 (2 pts): Cluster LBW Rate vs Mean Annual Temperature

Compute the Pearson correlation between cluster-level `lbw_rate` (from `zw_clusters.csv`) and mean annual `temp_c` (from the annual climate means). Store as `r_lbw_temp` (float, 3 decimal places).

Plot a scatter of the two variables with an OLS regression line.

In [ ]:
cluster_joint = (
    clusters[['cluster_id', 'lbw_rate']]
    .merge(clim_annual[['cluster_id', 'mean_temp']], on='cluster_id')
)

r_lbw_temp = round(float(cluster_joint['lbw_rate'].corr(cluster_joint['mean_temp'])), 3)
print(f"Pearson r (cluster LBW rate vs mean annual temperature): {r_lbw_temp:.3f}")

slope, intercept, *_ = stats.linregress(
    cluster_joint['mean_temp'], cluster_joint['lbw_rate']
)
x_line = np.linspace(cluster_joint['mean_temp'].min(), cluster_joint['mean_temp'].max(), 100)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    cluster_joint['mean_temp'], cluster_joint['lbw_rate'],
    s=18, alpha=0.5, color='steelblue', label='DHS cluster'
)
ax.plot(
    x_line, slope * x_line + intercept,
    color='firebrick', lw=2.0, label=f'OLS fit  (r = {r_lbw_temp})'
)
ax.set_xlabel('Mean annual 2 m air temperature (°C)', fontsize=11)
ax.set_ylabel('Cluster LBW rate', fontsize=11)
ax.set_title('Cluster-level LBW rate vs mean annual temperature\nZimbabwe DHS 2015', fontsize=12)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 We now have a dataset where every birth record has climate variables attached. What critical dimension of exposure is still missing?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q3" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> The province name</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q3" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> The cluster GPS coordinates</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q3" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> WHEN during the pregnancy each climate value applies — the same annual average hides trimester differences</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q3" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> The mother's age</span></label></div>
  <div id="nb04q3-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="C",E="Annual mean climate smooths over the timing of exposure within pregnancy. The next notebook addresses this: we will compute trimester-specific exposure windows.",ID="nb04q3";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Recap

| Concept | Key point |
|---|---|
| `cKDTree` | O(log n) nearest-neighbour search; build once on grid coords, query many points |
| DHS jitter | 0–5 km rural, 0–2 km urban, ≤10 km for 1% of rural — coordinates are never exact |
| Jitter uncertainty | Monte Carlo simulation: at ERA5/CHIRPS resolution, within-5km displacements often stay within one grid cell; uncertainty is detectable only with finer products (e.g., 1 km MODIS LST) |
| Exposure linkage | Merge births to annual mean climate on `cluster_id` — this is the exposure assignment step; individual GPS are unavailable in DHS by design |
| Spatial confounding | Facility proximity correlates with both LBW recording and local environment |

**Next:** Notebook 05 derives trimester-specific exposure windows from DHS birth dates, exploiting Zimbabwe's strong seasonal climate cycle to characterise *when* during pregnancy heat and drought occur.

In [ ]:
%%html
<div id="nb04-scoreboard" style="font-family:system-ui,sans-serif;border:2px solid #adb5bd;border-radius:10px;padding:20px 24px;margin:24px 0;background:#f8f9fa">
  <div style="font-size:17px;font-weight:700;margin-bottom:12px;color:#1a1a2e">&#128202; Module 4 &#8212; Spatial Join &#8212; Quiz Results</div>
  <div id="nb04-score-display"><p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Complete all reflection questions above to see your score.</p></div>
  <div id="nb04-next-action" style="margin-top:14px"></div>
</div>
<script>
(function(){
  var QUESTIONS=[{id:'nb04q1',correct:'B'},{id:'nb04q2',correct:'B'},{id:'nb04q3',correct:'C'}];
  var PASS=67;
  var NEXT='05_exposure_timeline_generation.ipynb';
  var NEXT_LABEL='Notebook 5: Exposure Timelines';
  var NBID='nb04';
  function update(){
    var correct=0,answered=0;
    QUESTIONS.forEach(function(q){
      var ans=localStorage.getItem('mcq_'+q.id);
      if(ans!==null){answered++;if(ans===q.correct)correct++;}
    });
    var total=QUESTIONS.length;
    var pct=total?Math.round(correct/total*100):0;
    var passed=pct>=PASS;
    var disp=document.getElementById(NBID+'-score-display');
    var act=document.getElementById(NBID+'-next-action');
    if(!disp)return;
    if(answered<total){
      disp.innerHTML='<p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Answer all '+total+' questions above to see your score. ('+(total-answered)+' remaining)</p>';
      if(act)act.innerHTML='';
    }else if(passed){
      disp.innerHTML='<div style="background:#d4edda;color:#155724;padding:12px 16px;border-radius:8px"><b>&#10003; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; MODULE PASSED</b><br><small>Your progress is saved &#8212; you can close and return to this notebook.</small></div>';
      if(act){
        localStorage.setItem(NBID+'_passed','true');
        if(NEXT){act.innerHTML='<a href="'+NEXT+'" style="display:inline-block;background:#198754;color:#fff;padding:10px 22px;border-radius:6px;text-decoration:none;font-weight:600;font-size:14px;margin-top:8px">&#10145; Continue to '+NEXT_LABEL+'</a>';}
        else{act.innerHTML='<div style="background:#cfe2ff;color:#084298;padding:12px 16px;border-radius:8px;margin-top:8px"><b>&#127881; You have completed the full GeoAI pipeline course! Congratulations.</b></div>';}
      }
    }else{
      disp.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:12px 16px;border-radius:8px"><b>&#10007; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; Below pass mark ('+PASS+'%)</b><br><small>Go back to any question and change your answer, then your score updates automatically.</small></div>';
      if(act)act.innerHTML='';
    }
  }
  document.addEventListener('mcq-update',update);
  if(document.readyState==='loading')document.addEventListener('DOMContentLoaded',update);
  else update();
  setInterval(update,3000);
})();
</script>

---

## ✅ Notebook 04 complete

Every birth record now has climate variables attached. You have completed the **climate exposure linkage** step — every birth record now carries the satellite-derived climate of its cluster. This is the core linking operation in environmental epidemiology.

**What is missing?** The climate values are annual averages. A heat anomaly in the first trimester matters differently than one at term.

➡️ **Notebook 05** reconstructs trimester-specific exposure windows from conception and birth dates, so the *timing* of environmental stress is captured.